# LLM overview
* here I am following Andrej Karpathy's video about llm - https://youtu.be/kCc8FmEb1nY?si=SfMfeMg16GxoU8hj
---

In [35]:
import requests
import torch
import torch.nn as nn
from torch.nn import functional # as F

In [29]:
RESET = "\033[0m"
BOLD = "\033[1m"

RED = "\033[31m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
BLUE = "\033[34m"
CYAN = "\033[36m"

def print_separator():
    print(f"{RED}================================================================================{RESET}")
    

### fetching tiny_shakespeare.

In [5]:
tiny_shakespeare_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
result = requests.get(tiny_shakespeare_url)

In [6]:
print(result)
type(result)
result??

<Response [200]>


Type:        Response
String form: <Response [200]>
File:        ~/ml/venv/lib/python3.14/site-packages/requests/models.py
Source:     
class Response:
    """The :class:`Response <Response>` object, which contains a
    server's response to an HTTP request.
    """

    _content: bytes | Literal[False] | None
    _content_consumed: bool
    _next: PreparedRequest | None
    status_code: int
    headers: CaseInsensitiveDict[str]
    raw: Any
    url: str
    encoding: str | None
    history: list[Response]
    reason: str
    cookies: RequestsCookieJar
    elapsed: datetime.timedelta
    request: PreparedRequest
    connection: HTTPAdapter

    __attrs__: list[str] = [
        "_content",
        "status_code",
        "headers",
        "url",
        "history",
        "encoding",
        "reason",
        "cookies",
        "elapsed",
        "request",
    ]

    def __init__(self) -> None:
        self._content = False
        self._content_consumed = False
        self._next = No

---
### extract the text

In [7]:
text = result._content.decode()
print(f"{BLUE}{BOLD}length of the dataset in characters: {len(text)}{RESET}\n")
print(text[:1000])

length of the dataset in characters: 1115394

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in 

### Take out the unique characters in the dataset

In [38]:
char = sorted(list(set(text)))
vocab_size = len(char)
print(f"{BLUE}{BOLD}all unique charactes:{RESET}{CYAN}{BOLD} {''.join(char)}{RESET}{BLUE}{BOLD}\nsize of the vocabulary:{RESET}{CYAN}{BOLD} {len(char)}{RESET}")

all unique charactes: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
size of the vocabulary: 65


the above characters contains newline character, hence it is in next line

---
### create mapping from characters to integers

In [9]:
stoi = {ch:i for i,ch in enumerate(char)}
itos = {i:ch for i,ch in enumerate(char)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"incoding -> {GREEN}{BOLD}{encode("hii there")}{RESET}")
print(f"decoding -> {GREEN}{BOLD}{decode(encode("hii there"))}{RESET}")

incoding -> [46, 47, 47, 1, 58, 46, 43, 56, 43]
decoding -> hii there


example tokenizers:
* SentencePiece - provides **BPE** *(Bypte Pair Encoding)* and **Kudo** *(unigram language model)*
* tiktoken - fast **BPE** tokenizer for use with openAI's models

### encode entire dataset and store in a tensor

In [10]:
data = torch.tensor(encode(text), dtype=torch.long)
print(f"{BLUE}{BOLD}shape = {RESET}{CYAN}{data.shape}{RESET}\n{BLUE}{BOLD}type = {RESET}{CYAN}{data.dtype}{RESET}\n{BLUE}{BOLD}data[:1000] = {RESET}{CYAN}{data[:1000]}{RESET}")

shape = torch.Size([1115394])
type = torch.int64
data[:1000] = tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 5

### Splitting data into **trainig** and **validiation** sets

In [11]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"{BLUE}{BOLD}size of train data = {RESET}{CYAN}{len(train_data)}{RESET}\n{BLUE}{BOLD}size of validation data = {RESET}{CYAN}{len(val_data)}{RESET}")

size of train data = 1003854
size of validation data = 111540


In [12]:
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [13]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"{BOLD}when input is {CYAN}{context}{RESET}{BOLD} the target is {CYAN}{target}{RESET}")
    

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [33]:
torch.manual_seed(1337) # I haven't changed the seed to confirm if the results are still in sync...
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y
    
print_separator()

xb, yb = get_batch('train')
print(f"{BOLD}inputs:{RESET}{CYAN}")
print(xb.shape)
print(xb)
print(f"{RESET}{BOLD}targets:{RESET}{CYAN}")
print(yb.shape)
print(yb)
print(f"{RESET}")

print_separator()

for b in range(batch_size):
    print("\n")
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"{BOLD}when input is {CYAN}{context}{RESET}{BOLD} the target is {CYAN}{target}{RESET}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])



when input is tensor([24]) the target is 43
when input is tensor([24, 43]) the target is 58
when input is tensor([24, 43, 58]) the target is 5
when input is tensor([24, 43, 58,  5]) the target is 57
when input is tensor([24, 43, 58,  5, 57]) the target is 1
when input is tensor([24, 43, 58,  5, 57,  1]) the target is 46
when input is tensor([24, 43, 58,  5, 57,  1, 46]) the target is 43
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target is 39


when input is tensor([44]) the target is 53
when input is tensor([44, 53]) the target is 56
when input is tensor([44, 53, 56]) the target is

In [47]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B,T,C)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = functional.cross_entropy(logits, targets)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:,-1,:]
            probs = functional.softmax(logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx



In [48]:
m = BigramLanguageModel(vocab_size)
out, loss = m(xb, yb)
print(out.shape)
print(loss)
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.1519, grad_fn=<NllLossBackward0>)

oCoV'ETzDUlW3D,q-B qkxdKHZt:VSNCXEzB-IPl.uotwf EOvyZ'DEnWq!mqkoMMm$'baSWMubK?DoIsRii$ga?WTd,rFKOVAM!
